In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/orders_clean.parquet"
)

print(df.shape)

(497128, 31)


In [2]:
baskets = (
    df.groupby("orderNumber")["skuNumber"]
    .apply(list)
    .tolist()
)

print("Total Orders:", len(baskets))

Total Orders: 177340


In [3]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()

basket_matrix = te.fit(
    baskets
).transform(
    baskets
)

basket_df = pd.DataFrame(
    basket_matrix,
    columns=te.columns_
)

print(basket_df.shape)

(177340, 252)


In [4]:
from mlxtend.frequent_patterns import fpgrowth

frequent_items = fpgrowth(
    basket_df,
    min_support=0.01,
    use_colnames=True
)

print(frequent_items.shape)

frequent_items.head()

(104, 2)


,support,itemsets
0,0.013539,frozenset({SKUBV8})
1,0.458453,frozenset({SKU00060})
2,0.447034,frozenset({SKU00086})
3,0.132480,frozenset({SKU00084})
4,0.032401,frozenset({SKU500})


In [5]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_items,
    metric="confidence",
    min_threshold=0.2
)

rules = rules.sort_values(
    "lift",
    ascending=False
)

print(rules.shape)

(92, 14)


In [6]:
rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
].head(20)

,antecedents,consequents,support,confidence,lift
38,"frozenset({SKU00033, SKU00086})","frozenset({SKU00084, SKU00060})",0.010539,0.437193,7.917872
40,"frozenset({SKU00033, SKU00060})","frozenset({SKU00084, SKU00086})",0.010539,0.425450,7.808054
66,frozenset({SKU613}),frozenset({SKU612}),0.018620,0.331593,7.401466
65,frozenset({SKU612}),frozenset({SKU613}),0.018620,0.415607,7.401466
23,"frozenset({SKU00084, SKU00060})","frozenset({SKUPM5, SKU00086})",0.019804,0.358660,6.555843
19,"frozenset({SKUPM5, SKU00086})","frozenset({SKU00084, SKU00060})",0.019804,0.361987,6.555843
20,"frozenset({SKU00084, SKU00086})","frozenset({SKUPM5, SKU00060})",0.019804,0.363448,6.469976
22,"frozenset({SKUPM5, SKU00060})","frozenset({SKU00084, SKU00086})",0.019804,0.352540,6.469976
52,frozenset({SKU00303}),frozenset({SKU00010}),0.014204,0.214529,3.718196
53,frozenset({SKU00010}),frozenset({SKU00303}),0.014204,0.246188,3.718196


In [7]:
sku_lookup = (
    df[
        [
            "skuNumber",
            "itemName"
        ]
    ]
    .drop_duplicates()
)

sku_dict = dict(
    zip(
        sku_lookup["skuNumber"],
        sku_lookup["itemName"]
    )
)

In [8]:
rules["antecedent_names"] = (
    rules["antecedents"]
    .apply(
        lambda x: [
            sku_dict.get(i, i)
            for i in x
        ]
    )
)

rules["consequent_names"] = (
    rules["consequents"]
    .apply(
        lambda x: [
            sku_dict.get(i, i)
            for i in x
        ]
    )
)

In [9]:
rules[
    [
        "antecedent_names",
        "consequent_names",
        "confidence",
        "lift"
    ]
].head(20)

,antecedent_names,consequent_names,confidence,lift
38,"[Rajnigandha 100g , Tulsi 00 (with Silver) 0.45g]","[Tulsi 00 4.25g (6 Pcs), Rajnigandha 4g]",0.437193,7.917872
40,"[Rajnigandha 100g , Rajnigandha 4g]","[Tulsi 00 4.25g (6 Pcs), Tulsi 00 (with Silver...",0.425450,7.808054
66,[Center Fresh],[Center Fruit],0.331593,7.401466
65,[Center Fruit],[Center Fresh],0.415607,7.401466
23,"[Tulsi 00 4.25g (6 Pcs), Rajnigandha 4g]","[Rajnigandha 17g 2 Pcs, Tulsi 00 (with Silver)...",0.358660,6.555843
19,"[Rajnigandha 17g 2 Pcs, Tulsi 00 (with Silver)...","[Tulsi 00 4.25g (6 Pcs), Rajnigandha 4g]",0.361987,6.555843
20,"[Tulsi 00 4.25g (6 Pcs), Tulsi 00 (with Silver...","[Rajnigandha 17g 2 Pcs, Rajnigandha 4g]",0.363448,6.469976
22,"[Rajnigandha 17g 2 Pcs, Rajnigandha 4g]","[Tulsi 00 4.25g (6 Pcs), Tulsi 00 (with Silver...",0.352540,6.469976
52,[RG Pearl Elaichi 1g Hanger],[Pass Pass Meetha Magic 11.75g Hanger],0.214529,3.718196
53,[Pass Pass Meetha Magic 11.75g Hanger],[RG Pearl Elaichi 1g Hanger],0.246188,3.718196


In [10]:
rules_save = rules.copy()

for col in [
    "antecedents",
    "consequents",
    "antecedent_names",
    "consequent_names"
]:
    rules_save[col] = (
        rules_save[col]
        .astype(str)
    )

rules_save.to_csv(
    "../data/processed/association_rules.csv",
    index=False
)

print("CSV Saved")

CSV Saved


In [11]:
rules_save.to_parquet(
    "../data/processed/association_rules.parquet",
    index=False
)

print("Parquet Saved")

Parquet Saved


In [12]:
print("Rules Shape:", rules.shape)

print(
    rules[
        [
            "antecedent_names",
            "consequent_names",
            "confidence",
            "lift"
        ]
    ].head(10)
)

Rules Shape: (92, 16)
                                     antecedent_names  \
38  [Rajnigandha 100g , Tulsi 00 (with Silver) 0.45g]   
40                [Rajnigandha 100g , Rajnigandha 4g]   
66                                     [Center Fresh]   
65                                     [Center Fruit]   
23           [Tulsi 00 4.25g (6 Pcs), Rajnigandha 4g]   
19  [Rajnigandha 17g 2 Pcs, Tulsi 00 (with Silver)...   
20  [Tulsi 00 4.25g (6 Pcs), Tulsi 00 (with Silver...   
22            [Rajnigandha 17g 2 Pcs, Rajnigandha 4g]   
52                       [RG Pearl Elaichi 1g Hanger]   
53             [Pass Pass Meetha Magic 11.75g Hanger]   

                                     consequent_names  confidence      lift  
38           [Tulsi 00 4.25g (6 Pcs), Rajnigandha 4g]    0.437193  7.917872  
40  [Tulsi 00 4.25g (6 Pcs), Tulsi 00 (with Silver...    0.425450  7.808054  
66                                     [Center Fruit]    0.331593  7.401466  
65                                    

In [13]:
print(rules.shape)

(92, 16)
